# Identifying 306 near-identical bronze statues

Wrocław is scattered with several hundred small bronze dwarf statues. They share a
vocabulary — the same scale, the same material, the same crouching poses — so telling one
from another is **instance recognition**, not classification: what separates them is a hat, a
tool, a pose.

This notebook reproduces the dataset's headline result from the published files alone, in
about twenty lines and a few seconds. No model is downloaded and no image is decoded: the
embeddings ship with the dataset.

- `images.csv` — one row per photograph, with attribution
- `embeddings_dinov2.npy` / `embeddings_clip.npy` — one row per photograph, **same order**
- `folds.csv` — the leave-one-out protocol

In [ ]:
import numpy as np
import pandas as pd

ROOT = "/kaggle/input/wroclaw-dwarves"

images = pd.read_csv(f"{ROOT}/images.csv")
classes = pd.read_csv(f"{ROOT}/classes.csv")
vectors = np.load(f"{ROOT}/embeddings_dinov2.npy")

print(f"{len(images):,} photographs of {len(classes)} statues")
print(f"vectors: {vectors.shape}, unit norm: {np.allclose(np.linalg.norm(vectors, axis=1), 1)}")
assert len(images) == len(vectors), "row i of the .npy is row i of images.csv"
images.head(3)

## The protocol

Leave-one-out: every photograph is queried in turn against **all the others**. A query is
correct when the highest-scoring *statue* is its own — collapsed to distinct statues by their
best-matching photograph, which is the candidate list a real tool would show.

The vectors are L2-normalised, so a dot product is a cosine.

In [ ]:
def score(vectors, labels, k=5):
    """Top-1, top-5 and MRR over distinct statues, leaving each query out."""
    similarity = vectors @ vectors.T
    np.fill_diagonal(similarity, -np.inf)  # never match yourself
    order = np.argsort(-similarity, axis=1)
    top1 = top5 = 0
    reciprocal = 0.0
    for query in range(len(labels)):
        ranked, seen = [], set()
        for candidate in order[query]:
            label = labels[candidate]
            if label not in seen:
                seen.add(label)
                ranked.append(label)
            if len(ranked) >= k and labels[query] in ranked:
                break
        position = ranked.index(labels[query]) + 1 if labels[query] in ranked else np.inf
        top1 += position == 1
        top5 += position <= k
        reciprocal += 1 / position
    n = len(labels)
    return {"top_1": top1 / n, "top_5": top5 / n, "mrr": reciprocal / n}


labels = images["label"].to_numpy()
for name in ("dinov2", "clip"):
    result = score(np.load(f"{ROOT}/embeddings_{name}.npy"), labels)
    print(
        f"{name:7} top-1 {result['top_1']:.1%}  "
        f"top-5 {result['top_5']:.1%}  MRR {result['mrr']:.3f}"
    )

**DINOv2 reaches 93.1% top-1 with no fine-tuning at all** — these are frozen, off-the-shelf
features. CLIP reaches 82.9% on exactly the same protocol, and the gap between them is what
most of this project's findings are about.

## What a retrieval actually looks like

The number above is an average over 1,691 of these.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image


def show(query_row, k=5):
    scores = vectors @ vectors[query_row]
    scores[query_row] = -np.inf
    best = {}
    for candidate in np.argsort(-scores):
        label = labels[candidate]
        if label not in best:
            best[label] = candidate
        if len(best) == k:
            break

    _, axes = plt.subplots(1, k + 1, figsize=(3 * (k + 1), 3.6))
    axes[0].imshow(Image.open(f"{ROOT}/{images.file_path[query_row]}"))
    axes[0].set_title(f"query\n{images.dwarf_name[query_row]}", fontweight="bold")
    for axis, (label, candidate) in zip(axes[1:], best.items(), strict=True):
        axis.imshow(Image.open(f"{ROOT}/{images.file_path[candidate]}"))
        correct = label == labels[query_row]
        axis.set_title(
            f"{images.dwarf_name[candidate]}\n{scores[candidate]:.3f}",
            color="#1f7a5c" if correct else "#cf4832",
        )
    for axis in axes:
        axis.set_xticks([])
        axis.set_yticks([])
    plt.tight_layout()
    plt.show()


show(0)

## Where it goes wrong, and why that is the interesting part

The errors are not random. They concentrate on families of near-identical statues that were
installed as themed groups — and `classes.csv` carries coordinates, so you can see that the
statues a model confuses are often the ones standing together.

In [ ]:
similarity = vectors @ vectors.T
np.fill_diagonal(similarity, -np.inf)
nearest = similarity.argmax(axis=1)
wrong = labels[nearest] != labels
print(f"nearest photograph is a different statue for {wrong.sum()} of {len(labels)} queries")

confusions = (
    pd.DataFrame(
        {
            "query": images.dwarf_name[wrong].to_numpy(),
            "mistaken_for": images.dwarf_name.to_numpy()[nearest[wrong]],
        }
    )
    .value_counts()
    .head(10)
)
confusions

## Attribution

Every photograph comes from Wikimedia Commons under its own licence, and this is a
**collection of separately licensed works** rather than one relicensed dataset. If you
redistribute any image, carry its credit: `images.csv` has `author`, `license` and
`license_url` per row, and `credits.csv` has a ready-to-paste line.

The statues themselves are contemporary sculptures under copyright. The Creative Commons
licences are granted by the photographers and cover the photographs only.

Full method, code and results: <https://github.com/turhancan97/krasnal-id>

In [ ]:
credits = pd.read_csv(f"{ROOT}/credits.csv")
print(f"{credits.author.nunique()} photographers")
print(credits.attribution_text.iloc[0])